In [1]:
import simplnx as nx

def check_pipeline_result(result: nx.Result) -> None:
  """
  This function will check the `result` for any errors. If errors do exist then a 
  `RuntimeError` will be thrown. Your own code to modify this to return something
  else that doesn't just stop your script in its tracks.
  """
  if len(result.warnings) != 0:
    for w in result.warnings:
      print(f'Warning: ({w.code}) {w.message}')
  
  has_errors = len(result.errors) != 0 
  if has_errors:
    #print(f'Pipeline :: Errors: {result.errors}')
    for err in result.errors:
      print(f'Error: ({err.code}) {err.message}')
    raise RuntimeError(result)
  
  print("Pipeline :: No errors running the pipeline")


![](Images/Tutorial_0/Slide_0.png)

# Slide 2: Expectations

- You already know python.
- Your IDE should be setup and ready to go
- We are going to write codes together in real time
- Feel free to sit back and just watch if you want


# Slide 3: Info, Code and Data

- [https://www.dream3d.io/blog/2025/07/28/2025-workshoponmethodsfor3dmicrostructurestudies/](https://www.dream3d.io/blog/2025/07/28/2025-workshoponmethodsfor3dmicrostructurestudies/)

- [https://www.dream3d.io/binaries/DREAM3DNXData.zip](https://www.dream3d.io/binaries/DREAM3DNXData.zip)
- Place it anywhere you want
- Open your IDE (VS Code/Spyder/PyCharm/etc)


# Slide 4: Environment Setup

```(console)
conda config --add channels conda-forge
conda config --set channel_priority strict
conda create -y -n nxpython python=3.12
conda activate nxpython
conda install -y -c bluequartzsoftware dream3dnx
conda install -y matplotlib scipy opencv pygraphviz
```

# Slide 5: Tools of the Trade

- HDFView: https://www.hdfgroup.org/downloads/hdfview/
- VS Code + ‘Excel Viewer’ for CSV Files
- VS Code + H5Web
    - Settings->’Workbench.EditorAssociations’, add “*.dream3d”:”h5web”

![](Images/Tutorial_0/Slide_5.png)

# Slide 6: What CAN'T I do with the Python Bindings (Today)

- Python bindings give you access to the non-gui portions of DREAM3D-NX
    - DataStructure, Filters, Plugin infrastructure
- Cannot drive the User Interface through Python
    - PyAutoGui maybe?
- Start the GUI pipeline running
    - Still needs “something” to push the run button


# Slide 7: What CAN I do then?

- Use Python to execute individual DREAM3D-NX Filters
- Load pipelines from file, possibly modify them, and then execute
- Create a “Filter” that can be loaded by the UI and is a first class citizen in DREAM3D-NX
- Programmatically create pipelines for execute and saving
- Extend #1 to import data from other sources
- Extend #1 to analyze DREAM3D resident data with other libraries
- Create a filter that simply wraps your existing python codes but presents a nice UI to the user?
- …. And a whole bunch of stuff that I haven’t thought of yet…


# Slide 9: Python Documentation

- Search for the filter within DREAM3D-NX
- Search for the filter on the web 
    - [https://www.dream3d.io/python_docs](https://www.dream3d.io/python_docs)

![](Images/Tutorial_0/Slide_9_1.png)

![](Images/Tutorial_0/Slide_9_2.png)

# Slide 10: Filter Documentation Guide

- [https://www.dream3d.io/python_docs/OrientationAnalysis.html#OrientationAnalysis.ReadAngDataFilter](https://www.dream3d.io/python_docs/OrientationAnalysis.html#OrientationAnalysis.ReadAngDataFilter)

- Many helpful links and tables to map from the UI to the Python API

![](Images/Tutorial_0/Slide_10.png)

# Slide 12: Just the Basics

##  DataStructure

- This is the main *Container* that holds all your DREAM3D Data. 
- Any other data created through python or read directly. 

## DataPath

Describes a path to get from the top of the DataStructure to a specific object within the data structure.

## DataArray

Holds the data.


# Slide 13: DataStructure Overview

## Represents a container that holds all the data objects created and/or processed during the execution of a pipeline.

## Data objects are stored in a hierarchical format, similar to files within directories on a file system

## These are the main types that are used in DREAM3D-NX

- Geometry
- Group
- AttributeMatrix
- DataArrays

## Usually a single DataStructure Object, but can have multiple if needed

## Only a single “DataStructure” can be written to a .dreamd file

![](Images/Tutorial_0/Slide_13.png)

In [2]:
# This code just loads an example DataStructure that we can use to showcase the APIs
data_structure = nx.DataStructure()
pipeline = nx.Pipeline.from_file('example_datastructure.d3dpipeline')
result = pipeline.execute(data_structure)
check_pipeline_result(result=result)

Pipeline :: No errors running the pipeline


# Slide 14: DataStructure API Examples

- Print the heirarchy to a string `hierarchy_to_str()`
- use the `[]` operator to retrieve the object at the given 'DataPath'
- Returns "DataArray" object that you cannot interact.

In [3]:
# 
print(f'{data_structure.hierarchy_to_str()}')

ci = data_structure["ImageGeometry/Cell Data/FeatureIds"]
print(f'ci object: {ci}')


|--Top_Level_Data
|--Data_Group
  |--uint16_data
  |--uint8_data
|--ImageGeometry
  |--Cell Data
    |--FeatureIds

ci object: <simplnx.Int32Array object at 0x118847e30>


## Slide 15: DataPath Overview

- Essentially a unique identifier for each data object within the DataStructure
- Used to reference and access objects within the DataStructure
- Specifies the location or the "path" to a specific data object, similar to how a file path works on a file system

![](Images/Tutorial_0/Slide_12.png)

In [4]:
my_path = nx.DataPath("Data_Group/uint16_data")
print(f'my_path: {my_path}')

my_path: Data_Group/uint16_data


## Slide 16: DataPath API Review:

The DataPath API mimics functions from the `PathLib` library.

`my_path = nx.DataPath( [“Group”,”SubGroup”,”data”])`

`my_path = nx.DataPath( “Group/SubGroup/data” )`

`to_string(“/”)`: converts DataPath to a string
`create_child_path()`: creates a child underneath the current path

`parts()`: Returns list of the parts of a path: 
    “Foo/Bar/Baz” => [“Foo”, “Bar”, “Baz”]

`parent()`: Returns the parent 
    “Foo/Bar/Baz” => “Foo/Bar”

`name()`: Returns the last part of the path
    “Foo/Bar/Baz” => “Baz”

`with_name(“Other”)`:
    “Foo/Bar/Baz” => “Foo/Bar/Other”


In [5]:
# Construct a DataPath from an "Array of Strings" or from a `/` delimited string
my_path = nx.DataPath(['ImageGeometry','Cell Data','FeatureIds']) 
my_path = nx.DataPath('ImageGeometry/Cell Data/FeatureIds')

# converts DataPath to a string
as_string = my_path.to_string('/')
print(f'as_string: {as_string}')

# creates a child underneath the current path
my_path = nx.DataPath("Data_Group")
my_path = my_path.create_child_path('More_Data')
print(f'my_path: {my_path}')

# get an array of strings representing each part of the path
# “Foo/Bar/Baz” => [“Foo”, “Bar”, “Baz”]
all_parts = my_path.parts()
print(f'all_parts: {all_parts}')

# : Returns the parent 
# “Foo/Bar/Baz” => “Foo/Bar”
parent_path = my_path.parent()
print(f'parent_path: {parent_path}')

# Returns the last part of the path
# “Foo/Bar/Baz” => “Baz”
name = my_path.name()
print(f'name: {name}')

# Change the 'Name' of a given path, i.e., change the last part
# “Foo/Bar/Baz” => “Foo/Bar/Other”
new_path = my_path.with_name('Other')
print(f'new_path: {new_path}')



as_string: ImageGeometry/Cell Data/FeatureIds
my_path: Data_Group/More_Data
all_parts: ['Data_Group', 'More_Data']
parent_path: Data_Group
name: More_Data
new_path: Data_Group/Other


# Slide 17: DataArray: How is your Data Stored

Each DataArray allocates a contiguous chunk of memory as requested by the developer or the runtime. All of the data for that DataArray is stored in this chunk of memory as a flat array. DREAM3D also uses the idea of a Tuple where each Tuple can have “N” number of components. Multiplying the number of Tuples by the number of Components gets you the total number of elements in an array.

As the developer you will need to calculate the correct index into the DataArray in order to get or set values. DataArray has methods to set/get based on Tuple, Component or Value. This slide shows various examples of setting a Tuple, Component or Value.

For example if you have an RGB image with 100 pixels, there are 100 Tuples, 3 Components, 300 total elements

![](Images/Tutorial_0/Slide_17.png)